# Mieloma múltiple — Factores al diagnóstico asociados a recaída tras la primera línea de tratamiento
**Healtech 2 · Universidad Externado de Colombia · Taller práctico de Ciencia de Datos en Salud (PBL)**

Este cuaderno reproduce **todos** los análisis del protocolo. Está organizado siguiendo las sesiones del taller y los documentos de la docente:

| Sección | Sesión del taller / documento guía |
|---|---|
| 1. Carga y estructura | Sesión 1 · Taller *Preparación y modelación de bases públicas* (Obj. 1) |
| 2. Calidad y limpieza | Sesión 2 · Taller bases públicas (Obj. 2, 3, 4, 6, 8) · HEATEC 1 Módulo 1 (higiene) |
| 3. Variables derivadas y temporales | Taller bases públicas (Obj. 3 y 7: transformación y *data abstraction*) |
| 4. Criterios de selección | Protocolo 6.5–6.6 · Taller bases públicas (Obj. 6: filtrado) |
| 5. Tabla 1 | Sesión 3 |
| 6. Análisis exploratorio gráfico | Sesión 4 · HEATEC 1 (validación gráfica post-limpieza) |
| 7. Normalidad y transformaciones | Sesión 5 · Taller 4 *Fase ANTES* (Shapiro-Wilk + Q-Q) |
| 8. Balance de clases y matrices X / Y | Taller bases públicas (Obj. 5) · HEATEC 1 (X_predictores.csv / Y_resultado.csv) |
| 9. Regresión lineal simple y múltiple (Hb) | Taller 4 *Experimentos A y B* + *Fase DESPUÉS* (residuos) |
| 10. Regresión logística simple y múltiple | Sesiones 6–8 · Taller 4 *Experimento C* |
| 11. Validación del modelo | Sesión 9 · Taller 4 Paso 3.2 (matriz de confusión, AUC) |
| 12. Análisis de sensibilidad y supervivencia | Sesión 9 (Kaplan–Meier, Cox, C-index) |
| 13. Exportar resultados | Producto final (base depurada, tablas, figuras) |

> **Antes de correr:** ejecute las celdas en orden (*Entorno de ejecución → Ejecutar todas*). Cuando aparezca el botón, suba el archivo `base organizada 06-07.2026.xlsx`.

In [ ]:
# ============================================================
# 0. CONFIGURACIÓN
# ============================================================
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

SEMILLA = 2026                      # semilla fija -> resultados reproducibles
np.random.seed(SEMILLA)
os.makedirs('resultados', exist_ok=True)   # aquí se guardan tablas y figuras

## 1. Carga de la base y estructura (Sesión 1)
La base tiene **una fila por paciente** y bloques de columnas repetidos para las líneas de tratamiento 1 a 5. Al leerla, pandas renombra las columnas repetidas agregando `.1`, `.2`, … Por eso **`recaida`, `trasplante`, `fechaRecaida`, `tieneLinea` (sin sufijo) corresponden a la PRIMERA línea**, `recaida.1` a la segunda, etc.

In [ ]:
# Carga del archivo: en Colab abre el selector de archivos; fuera de Colab usa la ruta local
try:
    from google.colab import files
    subido = files.upload()
    nombre = list(subido.keys())[0]
except ImportError:
    nombre = 'base organizada 06-07.2026.xlsx'

df_raw = pd.read_excel(nombre)
print('Dimensiones (pacientes, columnas):', df_raw.shape)

# Columnas de la primera línea de tratamiento (sin sufijo) y de la segunda (sufijo .1)
print([c for c in df_raw.columns if c.split('.')[0] in ['recaida', 'trasplante', 'tieneLinea', 'fechaRecaida']])

In [ ]:
# ---------- Funciones auxiliares ----------
def col(nombre_col):
    """Devuelve la columna si existe; si no, una columna vacía y un aviso (evita que el cuaderno se rompa)."""
    if nombre_col in df_raw.columns:
        return df_raw[nombre_col]
    print(f'AVISO: no existe la columna "{nombre_col}" -> se crea vacía')
    return pd.Series(np.nan, index=df_raw.index)

def a_numero(s):
    """Convierte texto con coma decimal ('2,79') a número; lo que no sea número queda NaN."""
    return pd.to_numeric(s.astype(str).str.strip().str.replace(',', '.', regex=False), errors='coerce')

def a_fecha(s):
    """Convierte fechas en formatos mixtos de la base: dd/mm/aaaa, aaaa-mm (fechas de tratamiento),
    fechas de Excel. Valores como '0-00', '0' o vacíos quedan como NaT (fecha faltante)."""
    if pd.api.types.is_datetime64_any_dtype(s):
        return s
    txt = s.astype(str).str.strip()
    out = pd.to_datetime(txt, format='%d/%m/%Y', errors='coerce')
    for fmt in ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d', '%Y-%m']:
        faltan = out.isna()
        out[faltan] = pd.to_datetime(txt[faltan], format=fmt, errors='coerce')
    return out

## 2. Calidad de los datos (Sesión 2)
Pasos: (a) duplicados, (b) seudonimización, (c) selección de variables de interés, (d) códigos de "sin dato", (e) rangos plausibles y unidades. Cada cambio queda contado en el **informe de calidad** (`resultados/informe_calidad.csv`).

In [ ]:
# (a) Duplicados: filas idénticas y posibles pacientes repetidos (mismas iniciales, nacimiento, sexo y fecha de diagnóstico)
print('Filas idénticas duplicadas:', df_raw.duplicated().sum())
clave = [c for c in ['iniciales', 'fechaNacimiento', 'sexo', 'fechaDx'] if c in df_raw.columns]
dup_clave = df_raw.duplicated(subset=clave, keep=False)
print('Registros con la misma clave de paciente', clave, ':', dup_clave.sum())
display(df_raw.loc[dup_clave, clave].sort_values(clave)) if dup_clave.any() else None

# Regla: se conserva el primer registro de cada paciente duplicado
df_raw = df_raw.loc[~df_raw.duplicated(subset=clave, keep='first')].reset_index(drop=True)
print('Pacientes después de eliminar duplicados:', len(df_raw))

In [ ]:
# (b) y (c) Seudonimización + selección de variables.
# No se conservan iniciales, usuario digitador ni fecha de nacimiento (Ley 1581 de 2012); se crea un código propio.
d = pd.DataFrame({'id_estudio': ['MM' + str(i + 1).zfill(4) for i in range(len(df_raw))]})

numericas_origen = ['idEnfermedad', 'edad', 'sexo', 'raza', 'regimen', 'ciudad',
                    'diaforesis', 'fiebre', 'sangrado', 'doloroseo', 'perpeso', 'infec', 'fractura', 'plasmo', 'frenal', 'dialisis',
                    'thb', 'hb', 'tcreatinina', 'creatinina', 'tcalcio', 'calcio', 'talbumina', 'albumina',
                    'tbeta2', 'beta2', 'tldh', 'ldh', 'tplaque', 'plaque', 'tleuco', 'leuco', 'plasmocitos',
                    'extramedular', 'iss', 'durie', 'resdel17p13', 'rest414', 'rest1416', 'estado', 'causaMuerte']
for v in numericas_origen:
    d[v] = a_numero(col(v)).values

d['centro'] = col('centro').values
# Primera línea de tratamiento (columnas sin sufijo)
d['tieneLinea_l1']  = a_numero(col('tieneLinea')).values
d['restto_l1']      = a_numero(col('restto')).values
d['trasplante_cod'] = a_numero(col('trasplante')).values
d['recaida_cod']    = a_numero(col('recaida')).values
# Fechas
d['fechaDx']        = a_fecha(col('fechaDx')).values
d['fechaSintomas']  = a_fecha(col('fechaSintomas')).values
d['fechaInicio_l1'] = a_fecha(col('fechaInicial')).values
d['fechaRecaida_l1']= a_fecha(col('fechaRecaida')).values
d['fechaControl']   = a_fecha(col('fechaControl')).values
print(d.shape)

In [ ]:
# (d) Códigos de "sin dato" según el diccionario de variables
informe = []   # aquí se registra cada corrección

def registrar(variable, regla, n):
    informe.append({'variable': variable, 'regla aplicada': regla, 'registros afectados': int(n)})

# Si la variable "¿hay dato?" (t...) vale 2 = No, el valor numérico no es válido aunque esté digitado (p. ej., Hb = 0)
pares = {'hb': 'thb', 'creatinina': 'tcreatinina', 'calcio': 'tcalcio', 'albumina': 'talbumina',
         'beta2': 'tbeta2', 'ldh': 'tldh', 'plaque': 'tplaque', 'leuco': 'tleuco'}
for v, t in pares.items():
    m = (d[t] == 2) & d[v].notna()
    registrar(v, f'{t}=2 (no hay dato) -> NaN', m.sum()); d.loc[m, v] = np.nan
    m = d[v] <= 0
    registrar(v, 'valor <= 0 (imposible) -> NaN', m.sum()); d.loc[m, v] = np.nan

# ISS: 1, 2, 3 son estadios; 4 = "Desconocido" en el diccionario (¡no es un estadio IV!); 0 = no diligenciado
m = ~d['iss'].isin([1, 2, 3]) & d['iss'].notna()
registrar('iss', 'ISS 4 (desconocido) o 0 -> NaN', m.sum())
d['iss_registrado'] = d['iss'].where(d['iss'].isin([1, 2, 3]))

# Durie-Salmon: 7 = desconocido; variables Si(1)/No(2): cualquier otro código -> NaN
d.loc[d['durie'] == 7, 'durie'] = np.nan
for v in ['diaforesis', 'fiebre', 'sangrado', 'doloroseo', 'perpeso', 'infec', 'fractura', 'plasmo', 'frenal', 'dialisis',
          'extramedular', 'resdel17p13', 'rest414', 'rest1416', 'trasplante_cod', 'recaida_cod', 'tieneLinea_l1']:
    m = ~d[v].isin([1, 2]) & d[v].notna()
    registrar(v, 'código distinto de 1/2 (0 = no diligenciado) -> NaN', m.sum()); d.loc[~d[v].isin([1, 2]), v] = np.nan
d.loc[~d['sexo'].isin([1, 2]), 'sexo'] = np.nan
d.loc[d['raza'] == 9, 'raza'] = np.nan

In [ ]:
# (e) Unidades y rangos plausibles.
# Unidades del diccionario: Hb g/dL, creatinina mg/dL, calcio mg/dL, albúmina g/dL, beta2 mg/L, plaquetas y leucocitos x10^3/mm3.
# Conversión SOLO cuando el patrón es claramente otra unidad (se reporta para verificación con la fuente):
reglas_unidad = [
    ('hb',         lambda x: (x >= 30) & (x <= 200),  lambda x: x / 10,   'Hb en g/L (30-200) -> g/dL (/10)'),
    ('albumina',   lambda x: (x >= 10) & (x <= 60),   lambda x: x / 10,   'Albúmina en g/L (10-60) -> g/dL (/10)'),
    ('creatinina', lambda x: (x > 20) & (x <= 1500),  lambda x: x / 88.4, 'Creatinina en µmol/L (>20) -> mg/dL (/88.4)'),
]
for v, cond, conv, texto in reglas_unidad:
    m = cond(d[v]).fillna(False)
    registrar(v, texto, m.sum()); d.loc[m, v] = conv(d.loc[m, v])

# Rangos fisiológicamente posibles; lo que quede por fuera se considera error de digitación -> NaN
RANGOS = {'edad': (18, 105), 'hb': (3, 20), 'creatinina': (0.2, 20), 'calcio': (5, 20), 'albumina': (1, 6.5),
          'beta2': (0.3, 100), 'ldh': (50, 5000), 'plaque': (5, 1500), 'leuco': (0.3, 300), 'plasmocitos': (0, 100)}
for v, (lo, hi) in RANGOS.items():
    m = (d[v] < lo) | (d[v] > hi)
    registrar(v, f'fuera de rango plausible [{lo}, {hi}] -> NaN', m.sum()); d.loc[m, v] = np.nan

# Consistencia de fechas: recaída antes del diagnóstico o control antes del diagnóstico = inconsistente
for f in ['fechaRecaida_l1', 'fechaControl', 'fechaInicio_l1']:
    m = (d[f] < d['fechaDx']).fillna(False)
    registrar(f, 'fecha anterior a fechaDx -> NaT', m.sum()); d.loc[m, f] = pd.NaT

informe = pd.DataFrame(informe)
informe = informe[informe['registros afectados'] > 0]
informe.to_csv('resultados/informe_calidad.csv', index=False, encoding='utf-8-sig')
informe

In [ ]:
# Porcentaje de datos faltantes por variable (después de la limpieza)
faltantes = (d.isna().mean() * 100).round(1).sort_values(ascending=False).rename('% faltante')
faltantes.to_csv('resultados/faltantes.csv', encoding='utf-8-sig')
faltantes[faltantes > 0]

## 3. Variables derivadas y temporales (transformación y *data abstraction*)
* **Desenlace (Y):** `recaida_l1` = 1 si el paciente presentó recaída/progresión después de la 1.ª línea; 0 si no.
* **ISS calculado** con los criterios de Greipp et al. (2005) a partir de β2-microglobulina y albúmina, para recuperar pacientes con ISS "desconocido" que sí tienen los laboratorios.
* **Criterios CRAB** (Rajkumar et al., 2014): anemia (Hb < 10 g/dL), falla renal (creatinina > 2 mg/dL), hipercalcemia (calcio > 11 mg/dL).
* **Transformación logarítmica** de variables sesgadas (creatinina, β2, LDH) → ver sección 7.
* **Tiempos** en meses desde el diagnóstico (para el análisis de supervivencia).

In [ ]:
d['recaida_l1']    = d['recaida_cod'].map({1: 1, 2: 0})      # Y binaria
d['trasplante_l1'] = d['trasplante_cod'].map({1: 1, 2: 0})
d['sexo_mujer']    = d['sexo'].map({2: 1, 1: 0})
d['del17p']        = d['resdel17p13'].map({1: 1, 2: 0})
for v in ['fractura', 'doloroseo', 'frenal', 'plasmo', 'extramedular']:      # Si(1)/No(2) -> 1/0
    d[v + '_si'] = d[v].map({1: 1, 2: 0})

# ISS calculado (Greipp 2005): I = β2 < 3.5 mg/L y albúmina >= 3.5 g/dL; III = β2 >= 5.5 mg/L; II = ni I ni III
b2, alb = d['beta2'], d['albumina']
iss_calc = pd.Series(np.nan, index=d.index)
iss_calc[(b2 >= 5.5)] = 3
iss_calc[(b2 < 3.5) & (alb >= 3.5)] = 1
iss_calc[(b2 >= 3.5) & (b2 < 5.5)] = 2
iss_calc[(b2 < 3.5) & (alb < 3.5)] = 2
d['iss_calc'] = iss_calc
d['iss_final'] = d['iss_registrado'].fillna(d['iss_calc'])
d['iss_fuente'] = np.where(d['iss_registrado'].notna(), 'registrado', np.where(d['iss_calc'].notna(), 'calculado', 'desconocido'))
print(d['iss_fuente'].value_counts())
print('Concordancia ISS registrado vs calculado (donde existen ambos):')
print(pd.crosstab(d['iss_registrado'], d['iss_calc']))
# Versión para análisis de sensibilidad: "desconocido" como categoría propia
d['iss_cat'] = d['iss_final'].map({1: 'I', 2: 'II', 3: 'III'}).fillna('Desconocido')

# Criterios CRAB al diagnóstico
d['anemia']         = np.where(d['hb'].notna(), (d['hb'] < 10).astype(float), np.nan)
d['creat_mayor2']   = np.where(d['creatinina'].notna(), (d['creatinina'] > 2).astype(float), np.nan)
d['hipercalcemia']  = np.where(d['calcio'].notna(), (d['calcio'] > 11).astype(float), np.nan)

# Transformaciones logarítmicas (variables con sesgo positivo)
for v in ['creatinina', 'beta2', 'ldh']:
    d['log_' + v] = np.log(d[v])

# Variables temporales (meses)
MES = 30.44
d['anio_dx'] = d['fechaDx'].dt.year
d['meses_sintomas_dx'] = (d['fechaDx'] - d['fechaSintomas']).dt.days / MES
d['meses_dx_inicio_tto'] = (d['fechaInicio_l1'] - d['fechaDx']).dt.days / MES
# Supervivencia libre de progresión (SLP): evento = recaída/progresión de la 1.ª línea o muerte (estado 9)
d['muerte'] = (d['estado'] == 9).astype(float).where(d['estado'].notna())
d['evento_slp'] = ((d['recaida_l1'] == 1) | (d['muerte'] == 1)).astype(float)
fecha_fin = d['fechaRecaida_l1'].where(d['recaida_l1'] == 1, d['fechaControl'])
d['meses_slp'] = (fecha_fin - d['fechaDx']).dt.days / MES
d.loc[d['meses_slp'] <= 0, 'meses_slp'] = np.nan

print('Periodo de diagnóstico en la base:', d['fechaDx'].min(), 'a', d['fechaDx'].max())
print('Centros:', d['centro'].value_counts(dropna=False).to_dict())

## 4. Criterios de inclusión y exclusión (diagrama de flujo)

In [ ]:
flujo = [('Registros en la base (sin duplicados)', len(d))]
b = d.copy()
b = b[b['idEnfermedad'] == 7];               flujo.append(('Diagnóstico de mieloma múltiple (idEnfermedad = 7)', len(b)))
b = b[b['edad'] >= 18];                      flujo.append(('Edad >= 18 años', len(b)))
b = b[b['tieneLinea_l1'] != 2];              flujo.append(('Recibió primera línea de tratamiento', len(b)))
base = b[b['recaida_l1'].notna()].copy();    flujo.append(('Con dato de recaída tras 1.ª línea (cohorte analítica)', len(base)))
flujo = pd.DataFrame(flujo, columns=['Paso', 'n'])
flujo.to_csv('resultados/flujo_pacientes.csv', index=False, encoding='utf-8-sig')
flujo

## 5. Tabla 1 — Características de la población (Sesión 3)
Variables continuas: media (DE) si Shapiro-Wilk p > 0,05; si no, mediana [RIC]. Categóricas: n (%). Comparación por desenlace: t de Student / U de Mann-Whitney y χ² / Fisher.

In [ ]:
CONTINUAS = ['edad', 'hb', 'creatinina', 'calcio', 'albumina', 'beta2', 'ldh', 'plaque', 'plasmocitos', 'meses_sintomas_dx']
CATEGORICAS = {'sexo_mujer': 'Sexo femenino', 'iss_cat': 'ISS', 'anemia': 'Anemia (Hb<10)', 'creat_mayor2': 'Creatinina >2 mg/dL',
               'hipercalcemia': 'Calcio >11 mg/dL', 'fractura_si': 'Fractura patológica', 'doloroseo_si': 'Dolor óseo',
               'frenal_si': 'Falla renal (registrada)', 'plasmo_si': 'Plasmocitoma',
               'del17p': 'del(17p) positiva', 'trasplante_l1': 'Trasplante como consolidación'}

def es_normal(x):
    x = x.dropna()
    return len(x) >= 3 and stats.shapiro(x)[1] > 0.05

def tabla1(df, y='recaida_l1'):
    filas = []
    g0, g1 = df[df[y] == 0], df[df[y] == 1]
    for v in CONTINUAS:
        x, a, c = df[v].dropna(), g0[v].dropna(), g1[v].dropna()
        if len(x) < 3: continue
        if es_normal(x):
            f = lambda s: f'{s.mean():.1f} ({s.std():.1f})'
            p = stats.ttest_ind(a, c, equal_var=False)[1] if len(a) > 1 and len(c) > 1 else np.nan
        else:
            f = lambda s: f'{s.median():.1f} [{s.quantile(.25):.1f}-{s.quantile(.75):.1f}]'
            p = stats.mannwhitneyu(a, c)[1] if len(a) > 0 and len(c) > 0 else np.nan
        filas.append([v, f'n={len(x)}', f(x), f(a) if len(a) else '', f(c) if len(c) else '', p])
    for v, etiqueta in CATEGORICAS.items():
        tab = pd.crosstab(df[v], df[y])
        p = np.nan
        if tab.shape == (2, 2):
            p = stats.fisher_exact(tab)[1] if (tab.values < 5).any() else stats.chi2_contingency(tab)[1]
        elif tab.shape[0] > 1 and tab.shape[1] == 2:
            p = stats.chi2_contingency(tab)[1]
        for nivel in sorted(df[v].dropna().unique(), key=str):
            if set(df[v].dropna().unique()) <= {0, 1} and nivel == 0: continue
            n_t = (df[v] == nivel).sum(); n0 = (g0[v] == nivel).sum(); n1 = (g1[v] == nivel).sum()
            rotulo = etiqueta if set(df[v].dropna().unique()) <= {0, 1} else f'{etiqueta}: {nivel}'
            filas.append([rotulo, f'n={df[v].notna().sum()}',
                          f'{n_t} ({100*n_t/df[v].notna().sum():.1f}%)',
                          f'{n0} ({100*n0/max(g0[v].notna().sum(),1):.1f}%)',
                          f'{n1} ({100*n1/max(g1[v].notna().sum(),1):.1f}%)', p])
            p = np.nan   # el p se muestra una sola vez por variable
    t = pd.DataFrame(filas, columns=['Variable', 'Datos disponibles', f'Total (n={len(df)})',
                                     f'Sin recaída (n={len(g0)})', f'Con recaída (n={len(g1)})', 'p'])
    t['p'] = t['p'].map(lambda x: '' if pd.isna(x) else ('<0.001' if x < 0.001 else f'{x:.3f}'))
    return t

T1 = tabla1(base)
T1.to_csv('resultados/tabla1.csv', index=False, encoding='utf-8-sig')
T1

## 6. Análisis exploratorio gráfico (Sesión 4 · validación gráfica HEATEC 1)
Histogramas y diagramas de caja para confirmar que la limpieza eliminó valores imposibles; barras para variables categóricas; dispersión y matriz de correlación (Spearman, porque varias variables no son normales).

In [ ]:
fig, ejes = plt.subplots(2, 6, figsize=(22, 7))
for i, v in enumerate(['edad', 'hb', 'creatinina', 'calcio', 'albumina', 'beta2']):
    ejes[0, i].hist(base[v].dropna(), bins=20, edgecolor='black'); ejes[0, i].set_title(f'Histograma: {v}')
    ejes[1, i].boxplot(base[v].dropna(), vert=True); ejes[1, i].set_title(f'Boxplot: {v}')
plt.tight_layout(); plt.savefig('resultados/fig1_histogramas_boxplots.png', dpi=150); plt.show()

fig, ejes = plt.subplots(1, 4, figsize=(18, 4))
for ax, v in zip(ejes, ['iss_cat', 'sexo_mujer', 'trasplante_l1', 'recaida_l1']):
    base[v].value_counts(dropna=False).sort_index().plot.bar(ax=ax, edgecolor='black'); ax.set_title(v)
plt.tight_layout(); plt.savefig('resultados/fig2_barras_categoricas.png', dpi=150); plt.show()

In [ ]:
# Hb según desenlace, y relación Hb - creatinina (anemia y daño renal son criterios CRAB)
fig, ejes = plt.subplots(1, 2, figsize=(12, 4.5))
base.boxplot(column='hb', by='recaida_l1', ax=ejes[0]); ejes[0].set_title('Hemoglobina según recaída (0 = no, 1 = sí)'); ejes[0].set_xlabel('')
ejes[1].scatter(base['log_creatinina'], base['hb'], alpha=.5)
ejes[1].set_xlabel('log(creatinina mg/dL)'); ejes[1].set_ylabel('Hb (g/dL)'); ejes[1].set_title('Hb vs log(creatinina)')
plt.suptitle(''); plt.tight_layout(); plt.savefig('resultados/fig3_hb.png', dpi=150); plt.show()

corr = base[['edad', 'hb', 'log_creatinina', 'calcio', 'albumina', 'log_beta2', 'log_ldh', 'plasmocitos', 'iss_final', 'recaida_l1']].corr(method='spearman')
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=60, ha='right'); ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.iloc[i, j]:.2f}', ha='center', va='center', fontsize=8)
plt.colorbar(im); plt.title('Matriz de correlación de Spearman'); plt.tight_layout()
plt.savefig('resultados/fig4_correlacion.png', dpi=150); plt.show()

## 7. Normalidad y transformaciones (Sesión 5 · Taller 4, *Fase ANTES*)
Regla del Taller 4: Shapiro-Wilk p > 0,05 → la variable se usa en su escala original; p < 0,05 → transformación logarítmica y nueva evaluación. Se complementa con el gráfico Q-Q, porque con muestras grandes la prueba rechaza normalidad ante desviaciones pequeñas.

In [ ]:
filas = []
for v in ['edad', 'hb', 'calcio', 'albumina', 'creatinina', 'log_creatinina', 'beta2', 'log_beta2', 'ldh', 'log_ldh', 'plasmocitos']:
    x = base[v].dropna()
    if len(x) < 3: continue
    w, p = stats.shapiro(x)
    filas.append([v, len(x), round(w, 3), p, 'Normal' if p > 0.05 else 'No normal', round(stats.skew(x), 2)])
normalidad = pd.DataFrame(filas, columns=['Variable', 'n', 'W', 'p', 'Decisión', 'Asimetría'])
normalidad['p'] = normalidad['p'].map(lambda x: '<0.001' if x < 0.001 else f'{x:.3f}')
normalidad.to_csv('resultados/normalidad.csv', index=False, encoding='utf-8-sig')
display(normalidad)

fig, ejes = plt.subplots(1, 4, figsize=(18, 4))
for ax, v in zip(ejes, ['hb', 'edad', 'creatinina', 'log_creatinina']):
    stats.probplot(base[v].dropna(), dist='norm', plot=ax); ax.set_title(f'Q-Q: {v}')
plt.tight_layout(); plt.savefig('resultados/fig5_qq.png', dpi=150); plt.show()

# Homogeneidad de varianzas (Levene) de Hb entre grupos del desenlace
a, c = base.loc[base.recaida_l1 == 0, 'hb'].dropna(), base.loc[base.recaida_l1 == 1, 'hb'].dropna()
print('Levene Hb (sin vs con recaída): p =', round(stats.levene(a, c)[1], 3))

## 8. Balance de clases y separación de matrices X / Y
No se aplica sobremuestreo (SMOTE) ni submuestreo: en modelos de riesgo estas correcciones distorsionan la calibración sin mejorar el AUC (van den Goorbergh et al., 2022). El desbalance se maneja eligiendo un **punto de corte** adecuado en la validación (sección 11).

In [ ]:
print(base['recaida_l1'].value_counts().rename({0: 'Sin recaída', 1: 'Con recaída'}))
print('Prevalencia del desenlace: %.1f%%' % (100 * base['recaida_l1'].mean()))

PREDICTORES = ['edad', 'iss_final', 'hb', 'trasplante_l1']          # modelo principal preespecificado
EXTRA = ['sexo_mujer', 'log_creatinina', 'calcio']                  # modelo extendido (exploratorio)
base[['id_estudio'] + PREDICTORES + EXTRA].to_csv('resultados/X_predictores.csv', index=False, encoding='utf-8-sig')
base[['id_estudio', 'recaida_l1']].to_csv('resultados/Y_resultado.csv', index=False, encoding='utf-8-sig')
print('Guardados X_predictores.csv y Y_resultado.csv')

## 9. Regresión lineal: hemoglobina al diagnóstico (Taller 4, Experimentos A y B + Fase DESPUÉS)
Objetivo secundario: explorar factores asociados a la Hb (Y continua). Predictor principal: log(creatinina) (daño renal → menor producción de eritropoyetina); se ajusta por % de plasmocitos en médula (infiltración medular), edad y sexo.

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import het_breuschpagan

d_lin = base[['hb', 'log_creatinina', 'plasmocitos', 'edad', 'sexo_mujer']].dropna()
print('Pacientes con datos completos para el modelo lineal:', len(d_lin))

# Experimento A: regresión lineal simple
m_simple = smf.ols('hb ~ log_creatinina', data=d_lin).fit()
print(m_simple.summary().tables[1]); print('R2 =', round(m_simple.rsquared, 3))

# Experimento B: regresión lineal múltiple
m_mult = smf.ols('hb ~ log_creatinina + plasmocitos + edad + sexo_mujer', data=d_lin).fit()
print(m_mult.summary().tables[1]); print('R2 ajustado =', round(m_mult.rsquared_adj, 3))

In [ ]:
# Fase DESPUÉS: auditoría de residuos
res = m_mult.resid
w, p_res = stats.shapiro(res)
bp = het_breuschpagan(res, m_mult.model.exog)
print(f'Shapiro-Wilk de residuos: p = {p_res:.4f} ->', 'MODELO APROBADO (residuos normales)' if p_res > 0.05 else 'residuos NO normales: revisar')
print(f'Breusch-Pagan (heterocedasticidad): p = {bp[1]:.4f}')

fig, ejes = plt.subplots(1, 3, figsize=(17, 4))
ejes[0].hist(res, bins=20, edgecolor='black'); ejes[0].set_title('Histograma de residuos')
sm.qqplot(res, line='s', ax=ejes[1]); ejes[1].set_title('Q-Q de residuos')
ejes[2].scatter(m_mult.fittedvalues, res, alpha=.5); ejes[2].axhline(0, color='red')
ejes[2].set_xlabel('Valores ajustados'); ejes[2].set_ylabel('Residuo'); ejes[2].set_title('Residuos vs ajustados (¿forma de U o abanico?)')
plt.tight_layout(); plt.savefig('resultados/fig6_residuos_lineal.png', dpi=150); plt.show()

## 10. Regresión logística (Sesiones 6–8 · Taller 4, Experimento C)
**Ruta metodológica:** factores de riesgo / clasificación → Y binaria → regresión logística.

**Modelo principal preespecificado** (variables elegidas por evidencia clínica, no por valor p): edad, ISS, hemoglobina y trasplante como consolidación. Se limita a 5 parámetros para respetar ~10 eventos por variable (EPV; Peduzzi et al., 1996).

In [ ]:
def tabla_or(modelo):
    ic = modelo.conf_int()
    t = pd.DataFrame({'OR': np.exp(modelo.params), 'IC95% inf': np.exp(ic[0]), 'IC95% sup': np.exp(ic[1]), 'p': modelo.pvalues})
    return t.drop(index='Intercept').round(3)

# (a) Análisis bivariado: OR crudos (logística simple), cada uno con sus datos disponibles
crudos = []
for v in ['edad', 'sexo_mujer', 'C(iss_final)', 'hb', 'log_creatinina', 'calcio', 'albumina', 'log_beta2', 'trasplante_l1', 'del17p']:
    nombre_v = v.replace('C(', '').replace(')', '')
    dd = base[['recaida_l1', nombre_v]].dropna()
    if dd[nombre_v].nunique() < 2 or dd['recaida_l1'].sum() < 5: continue
    m = smf.logit(f'recaida_l1 ~ {v}', data=dd).fit(disp=0)
    t = tabla_or(m); t.insert(0, 'n', len(dd)); crudos.append(t)
crudos = pd.concat(crudos)
crudos.to_csv('resultados/OR_crudos.csv', encoding='utf-8-sig')
crudos

In [ ]:
# (b) Modelo multivariable principal (casos completos)
datos = base[['id_estudio', 'recaida_l1'] + PREDICTORES].dropna().copy()
eventos = int(datos['recaida_l1'].sum())
print(f'Pacientes: {len(datos)} | eventos: {eventos} | parámetros: 5 | EPV = {eventos/5:.1f}')

FORMULA = 'recaida_l1 ~ edad + C(iss_final) + hb + trasplante_l1'
modelo = smf.logit(FORMULA, data=datos).fit(disp=0)
print(modelo.summary())
tabla_modelo = tabla_or(modelo)
tabla_modelo.to_csv('resultados/OR_modelo_principal.csv', encoding='utf-8-sig')
tabla_modelo

In [ ]:
# Multicolinealidad (VIF > 5 es preocupante)
from statsmodels.stats.outliers_influence import variance_inflation_factor
X = modelo.model.exog
print(pd.Series([variance_inflation_factor(X, i) for i in range(1, X.shape[1])], index=modelo.model.exog_names[1:], name='VIF').round(2))
# Prueba de razón de verosimilitud global
print(f'LR chi2 = {modelo.llr:.2f}, p = {modelo.llr_pvalue:.4f}; pseudo-R2 de McFadden = {modelo.prsquared:.3f}')
# Comparación de pacientes incluidos vs excluidos por datos faltantes (sesgo de selección)
base['incluido_modelo'] = base['id_estudio'].isin(datos['id_estudio']).astype(int)
print(base.groupby('incluido_modelo')[['edad', 'recaida_l1', 'trasplante_l1']].mean().round(2))

## 11. Validación del modelo (Sesión 9)
* **Discriminación:** AUC aparente y AUC corregido por optimismo con *bootstrap* (200 remuestreos) + validación cruzada estratificada 5×10.
* **Calibración:** pendiente e intercepto de calibración, gráfico de calibración, Hosmer-Lemeshow y Brier.
* **Clasificación:** matriz de confusión, sensibilidad, especificidad, VPP (precisión) y VPN con tres puntos de corte: 0,5; la prevalencia; y el índice de Youden.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, brier_score_loss
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score

def matriz_X(df):
    X = df[['edad', 'hb', 'trasplante_l1']].copy()
    for k in [2, 3]:
        X[f'iss_{k}'] = (df['iss_final'] == k).astype(int)    # ISS I = referencia
    return X.values

Xv, yv = matriz_X(datos), datos['recaida_l1'].values.astype(int)
lr = LogisticRegression(penalty=None, max_iter=5000)     # sin penalización = mismos coeficientes que el modelo de la sección 10
lr.fit(Xv, yv)
p_hat = lr.predict_proba(Xv)[:, 1]

def calib(y, p):
    """Pendiente e intercepto de calibración (logística de y sobre el logit de p)."""
    lp = np.log(p / (1 - p)).reshape(-1, 1)
    c = LogisticRegression(penalty=None).fit(lp, y)
    return c.coef_[0][0], c.intercept_[0]

auc_ap = roc_auc_score(yv, p_hat)
pend_ap, _ = calib(yv, p_hat)

# Bootstrap de optimismo (Harrell / Steyerberg)
rng = np.random.default_rng(SEMILLA)
opt_auc, opt_pend = [], []
for _ in range(200):
    idx = rng.integers(0, len(yv), len(yv))
    if yv[idx].sum() < 2: continue
    mb = LogisticRegression(penalty=None, max_iter=5000).fit(Xv[idx], yv[idx])
    pb_boot, pb_orig = mb.predict_proba(Xv[idx])[:, 1], mb.predict_proba(Xv)[:, 1]
    opt_auc.append(roc_auc_score(yv[idx], pb_boot) - roc_auc_score(yv, pb_orig))
    opt_pend.append(calib(yv[idx], pb_boot)[0] - calib(yv, pb_orig)[0])

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=SEMILLA)
auc_cv = cross_val_score(LogisticRegression(penalty=None, max_iter=5000), Xv, yv, cv=cv, scoring='roc_auc')

resumen = pd.Series({
    'AUC aparente': auc_ap,
    'Optimismo AUC (bootstrap)': np.mean(opt_auc),
    'AUC corregido por optimismo': auc_ap - np.mean(opt_auc),
    'AUC validación cruzada 5x10 (media)': auc_cv.mean(),
    'AUC validación cruzada (DE)': auc_cv.std(),
    'Pendiente de calibración corregida (factor de encogimiento)': pend_ap - np.mean(opt_pend),
    'Brier': brier_score_loss(yv, p_hat),
    'Brier de referencia (prevalencia)': brier_score_loss(yv, np.repeat(yv.mean(), len(yv))),
}).round(3)
resumen.to_csv('resultados/validacion_resumen.csv', encoding='utf-8-sig')
resumen

In [ ]:
# Curva ROC e intervalo de confianza del AUC por bootstrap
aucs = []
for _ in range(1000):
    idx = rng.integers(0, len(yv), len(yv))
    if 0 < yv[idx].sum() < len(idx): aucs.append(roc_auc_score(yv[idx], p_hat[idx]))
print(f'AUC aparente = {auc_ap:.3f} (IC95% bootstrap {np.percentile(aucs, 2.5):.3f} - {np.percentile(aucs, 97.5):.3f})')

fpr, tpr, umbrales = roc_curve(yv, p_hat)
plt.figure(figsize=(5.5, 5)); plt.plot(fpr, tpr, label=f'Modelo (AUC = {auc_ap:.2f})'); plt.plot([0, 1], [0, 1], '--', color='gray', label='Azar (0.5)')
plt.xlabel('1 - Especificidad'); plt.ylabel('Sensibilidad'); plt.title('Curva ROC'); plt.legend(); plt.tight_layout()
plt.savefig('resultados/fig7_roc.png', dpi=150); plt.show()

In [ ]:
# Calibración: observados vs predichos por quintiles de riesgo + Hosmer-Lemeshow
cal = pd.DataFrame({'y': yv, 'p': p_hat})
cal['grupo'] = pd.qcut(cal['p'], q=5, duplicates='drop')
g = cal.groupby('grupo', observed=True).agg(predicho=('p', 'mean'), observado=('y', 'mean'), n=('y', 'size'), eventos=('y', 'sum'))
display(g.round(3))
hl = (((g['eventos'] - g['n'] * g['predicho']) ** 2) / (g['n'] * g['predicho'] * (1 - g['predicho']))).sum()
print(f'Hosmer-Lemeshow (quintiles): chi2 = {hl:.2f}, p = {1 - stats.chi2.cdf(hl, len(g) - 2):.3f}')

plt.figure(figsize=(5.5, 5)); plt.plot(g['predicho'], g['observado'], 'o-', label='Quintiles')
lim = max(g['predicho'].max(), g['observado'].max()) * 1.1
plt.plot([0, lim], [0, lim], '--', color='gray', label='Calibración perfecta')
plt.xlabel('Riesgo predicho'); plt.ylabel('Proporción observada de recaída'); plt.title('Gráfico de calibración'); plt.legend(); plt.tight_layout()
plt.savefig('resultados/fig8_calibracion.png', dpi=150); plt.show()

In [ ]:
# Matriz de confusión y métricas con tres puntos de corte
j = np.argmax(tpr - fpr)
cortes = {'0.5 (por defecto)': 0.5, 'Prevalencia': yv.mean(), 'Youden': umbrales[j]}
filas = []
for nombre_c, u in cortes.items():
    pred = (p_hat >= u).astype(int)
    tn, fp, fn, tp = confusion_matrix(yv, pred, labels=[0, 1]).ravel()
    filas.append({'Punto de corte': f'{nombre_c} = {u:.3f}', 'VP': tp, 'FP': fp, 'FN': fn, 'VN': tn,
                  'Sensibilidad': tp / (tp + fn) if tp + fn else np.nan, 'Especificidad': tn / (tn + fp) if tn + fp else np.nan,
                  'VPP (precisión)': tp / (tp + fp) if tp + fp else np.nan, 'VPN': tn / (tn + fn) if tn + fn else np.nan,
                  'Exactitud': (tp + tn) / len(yv)})
clasif = pd.DataFrame(filas).round(3)
clasif.to_csv('resultados/clasificacion_puntos_corte.csv', index=False, encoding='utf-8-sig')
clasif

## 12. Análisis de sensibilidad y de supervivencia
1. **ISS desconocido como categoría** (para no perder los eventos de pacientes sin estadificación).
2. **Modelo extendido** (agrega sexo, log-creatinina y calcio) — exploratorio porque baja el EPV.
3. **Supervivencia libre de progresión (SLP):** la logística no considera cuánto tiempo se siguió a cada paciente; un paciente con seguimiento corto puede figurar como "sin recaída" solo porque no alcanzó a recaer. Kaplan–Meier, log-rank, Cox e índice C corrigen ese problema.

In [ ]:
# 1) ISS desconocido como categoría propia
d_s1 = base[['recaida_l1', 'edad', 'iss_cat', 'hb', 'trasplante_l1']].dropna()
m_s1 = smf.logit("recaida_l1 ~ edad + C(iss_cat, Treatment('I')) + hb + trasplante_l1", data=d_s1).fit(disp=0)
print('Sensibilidad 1 - n =', len(d_s1), '| eventos =', int(d_s1.recaida_l1.sum()))
display(tabla_or(m_s1))

# 2) Modelo extendido
d_s2 = base[['recaida_l1'] + PREDICTORES + EXTRA].dropna()
m_s2 = smf.logit(FORMULA + ' + sexo_mujer + log_creatinina + calcio', data=d_s2).fit(disp=0)
print('Sensibilidad 2 - n =', len(d_s2), '| eventos =', int(d_s2.recaida_l1.sum()), '| EPV =', round(d_s2.recaida_l1.sum() / 8, 1))
display(tabla_or(m_s2))

In [ ]:
# 3) Supervivencia libre de progresión: Kaplan-Meier, log-rank y Cox
from statsmodels.duration.survfunc import SurvfuncRight, survdiff

d_surv = base[['meses_slp', 'evento_slp', 'edad', 'iss_final', 'hb', 'trasplante_l1']].dropna()
print('Pacientes con tiempo de seguimiento válido:', len(d_surv), '| eventos:', int(d_surv.evento_slp.sum()))

km = SurvfuncRight(d_surv['meses_slp'], d_surv['evento_slp'])
print('Mediana de SLP (meses):', km.quantile(0.5))
fig, ax = plt.subplots(figsize=(7, 5))
for k in [1, 2, 3]:
    s = d_surv[d_surv.iss_final == k]
    if len(s) > 0:
        SurvfuncRight(s['meses_slp'], s['evento_slp']).plot(ax=ax)
lineas = [l for l in ax.get_lines() if l.get_linestyle() != 'None']    # se excluyen las marcas de censura
ax.legend(lineas, [f'ISS {k}' for k in [1, 2, 3] if (d_surv.iss_final == k).any()])
ax.set_xlabel('Meses desde el diagnóstico'); ax.set_ylabel('Probabilidad libre de progresión'); ax.set_title('Kaplan-Meier por ISS')
plt.tight_layout(); plt.savefig('resultados/fig9_km_iss.png', dpi=150); plt.show()
chi2, p_lr = survdiff(d_surv['meses_slp'], d_surv['evento_slp'], d_surv['iss_final'])
print(f'Log-rank por ISS: chi2 = {chi2:.2f}, p = {p_lr:.4f}')

In [ ]:
# Modelo de Cox (mismos predictores que el modelo logístico principal)
cox = smf.phreg('meses_slp ~ edad + C(iss_final) + hb + trasplante_l1', data=d_surv, status=d_surv['evento_slp'].values).fit()
ic, coef = np.asarray(cox.conf_int()), np.asarray(cox.params)
tabla_cox = pd.DataFrame({'HR': np.exp(coef), 'IC95% inf': np.exp(ic[:, 0]), 'IC95% sup': np.exp(ic[:, 1]), 'p': np.asarray(cox.pvalues)},
                         index=cox.model.exog_names).round(3)
tabla_cox.to_csv('resultados/HR_cox.csv', encoding='utf-8-sig')
display(tabla_cox)

# Índice de concordancia de Harrell (C-index), calculado a partir del predictor lineal
def c_index(t, e, riesgo):
    t, e, riesgo = np.asarray(t), np.asarray(e), np.asarray(riesgo)
    conc = perm = 0.0
    for i in range(len(t)):
        if e[i] != 1: continue
        comparables = t > t[i]
        perm += comparables.sum()
        conc += (riesgo[i] > riesgo[comparables]).sum() + 0.5 * (riesgo[i] == riesgo[comparables]).sum()
    return conc / perm

lp = cox.model.exog @ coef
print(f'C-index (aparente) del modelo de Cox: {c_index(d_surv.meses_slp, d_surv.evento_slp, lp):.3f}')

## 13. Exportar base depurada y resultados (producto final)

In [ ]:
base.to_csv('resultados/base_depurada_analitica.csv', index=False, encoding='utf-8-sig')
d.to_csv('resultados/base_depurada_completa.csv', index=False, encoding='utf-8-sig')
!zip -qr resultados.zip resultados
try:
    files.download('resultados.zip')
except Exception:
    print('Archivos guardados en la carpeta resultados/')